# Day 2, hands-on 1: trace the calls, worked

Four functions, and the only question that matters is what each one hands back to its caller.

Every placeholder is filled with the option the answer key records, and the notebook is executed
from a clean kernel so every output and every check is visible on the page. The line under each
step says why the other three letters fail.

Where this sits in the day, and the steps this notebook walks.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["functions and errors", "files and formats", "hands-on: trace the calls", "hands-on: the truncated feed"], lit=2, title="the day's notebooks", show=False),
    kit.flow(["what print hands back", "what return hands back", "chain two calls", "the caller decides"], title="this notebook's steps", show=False),
)

## Setup

The thirty orders from Tuesday's CSV, read straight from `../data/`. Everything a CSV gives you is text, which is why the conversion questions below are real.

In [2]:
import csv

orders = kit.load_csv("C2_W01_D02_orders_STUDENT.csv")

print(len(orders), "rows read from ../data/")
print(orders[0])

30 rows read from ../data/
{'order_id': 'KR4200', 'customer_id': 'C1645', 'segment': 'Retail-Core', 'amount': '4500', 'status': 'returned', 'order_date': '2026-08-03', 'discount': ''}


## Step 1. A function that only prints

`show_id` prints and does nothing else. Decide what its caller is holding after the call, before you run anything.

In [3]:
kit.flow(["what print hands back", "what return hands back", "chain two calls", "the caller decides"], lit=0)

In [4]:
# TODO 1. What is the caller holding in result?
#   a) the order id, as text
#   b) None, since nothing was returned
#   c) the record itself
#   d) an error, since print returns nothing
def show_id(record):
    print(record["order_id"])

result = show_id(orders[0])
held = repr(result)
print("held:", held)

KR4200
held: None


In [5]:
kit.check("a print-only function hands back None", result is None)
kit.check("held names what the caller actually has", held == "None", f"held is {held}")

Every option renders the same expression on purpose, so the letter records what you believed rather than changing what runs. Option a is the value you saw on screen, which the caller never received. Option c confuses the argument with the return. Option d treats a missing return as an error, when Python supplies `None` silently, which is exactly what makes this failure hard.

## Step 2. A function that returns

The same work, with the answer handed back instead of shown. The difference is one keyword and it decides whether the next line can do anything.

In [6]:
kit.flow(["what print hands back", "what return hands back", "chain two calls", "the caller decides"], lit=1)

In [7]:
# TODO 2. Which expression proves the caller can use what came back?
#   a) returned is None
#   b) print(returned)
#   c) isinstance(returned, str)
#   d) len(orders) > 0
def get_id(record):
    return record["order_id"]

returned = get_id(orders[0])
usable = isinstance(returned, str)
print(returned, usable)

KR4200 True


In [8]:
kit.check("the caller holds the id itself", returned == orders[0]["order_id"])
kit.check("and it is text it can pass on", usable is True)

Option a is true of the previous step and false here. Option b shows the value again rather than testing it, which is the confusion the whole step exists to break. Option d tests the file rather than the return.

## Step 3. Chain two calls

A returned value is only worth having if the next call can take it. Convert an amount, then hand the result to something that needs a number.

In [9]:
kit.flow(["what print hands back", "what return hands back", "chain two calls", "the caller decides"], lit=2)

In [10]:
# TODO 3. What has to happen between the two calls?
#   a) normalise_amount(raw)
#   b) str(raw)
#   c) raw
#   d) is_large(raw)
def normalise_amount(raw):
    return int(raw)

def is_large(amount):
    return amount > 2000

raw = orders[1]["amount"]
converted = normalise_amount(raw)
verdict = is_large(converted)
print(raw, converted, verdict)

2395 2395 True


In [11]:
kit.check("the amount arrived as text and left as a number",
          isinstance(raw, str) and isinstance(converted, int))
kit.check("the second call could answer at all", isinstance(verdict, bool))

Option b keeps it as text, so `is_large` compares a string against a number and raises. Option c passes the text straight through with the same result. Option d calls the second function on the raw value and skips the conversion entirely.

## Step 4. The caller decides what a failure means

`normalise_amount` raises on the two planted amounts. The function has no opinion about what that means for the run; the loop around it does.

In [12]:
kit.flow(["what print hands back", "what return hands back", "chain two calls", "the caller decides"], lit=3)

In [13]:
# TODO 4. What belongs in the rejects list?
#   a) r["order_id"]
#   b) str(e)
#   c) the id and the interpreter's own reason
#   d) nothing, since it is being skipped
def normalise_amount(raw):
    return int(raw)

clean, rejects = [], []
for r in orders:
    try:
        clean.append(normalise_amount(r["amount"]))
    except ValueError as e:
        rejects.append({"order_id": r["order_id"], "reason": str(e)})

print(len(clean), len(rejects))
print(rejects[0])

28 2
{'order_id': 'KR4210', 'reason': "invalid literal for int() with base 10: 'twelve'"}


In [14]:
kit.check("28 of the 30 amounts converted", len(clean) == 28, f"{len(clean)} clean")
kit.check("two were rejected", len(rejects) == 2)
kit.check("every rejection names the record and the reason",
          all(set(x) == {"order_id", "reason"} for x in rejects))

Every option renders the same dictionary, so the letter records your reasoning. An id alone cannot be acted on, a reason alone cannot be traced to a record, and skipping silently is the bare-except failure in a different costume.

## What to post

Post one line with the four letters, then the reconciliation:

```
1b 2c 3a 4c
30 in = 28 clean + 2 rejected
```

Then one sentence naming the two rejected order ids and what each reason says.

In [15]:
kit.flow(["what print hands back", "what return hands back", "chain two calls", "the caller decides"], lit=3, title="the notebook, end to end")
kit.check_summary()